# Práctica 1 - Predicción de Subscripción a un Producto Bancario

**Asignatura:** Aprendizaje Automático 2025-26  
**Grupo:**
- Pablo García Aparicio
- Miguel Merino Sánchez

**NIA:** 100522190  
**Fichero de datos:** `bank_09.pkl`

---
> **Nota sobre IA generativa:** En esta practica hemos usadado la IA generativa para ayudarnos a comporbar el codigo base de pipelines y verificar la logica de los preprocesamientos.

In [2]:
# ─── Imports globales ───────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import time
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder, LabelEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

# Semilla reproducibilidad = NIA
SEED = 100522190
np.random.seed(SEED)

# Estilo gráficos
sns.set_theme(style='whitegrid', palette='muted')
matplotlib.rcParams['figure.dpi'] = 100

print('Librerías cargadas correctamente.')

Librerías cargadas correctamente.


## 1. Carga de datos

In [3]:
df = pd.read_pickle('bank_09.pkl')
print(f'Dataset cargado: {df.shape[0]} instancias, {df.shape[1]} variables')
df.head()

Dataset cargado: 11000 instancias, 17 variables


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


---
## 2. EDA Simplificado

Analizamos de forma sistemática y mediante código el dataset para guiar el preprocesamiento.


In [4]:
# ── 2.1 Dimensiones y tipos ───────────────────────────────────────────────
print(f'Filas: {df.shape[0]}, Columnas: {df.shape[1]}')
print('\n--- Tipos de datos ---')
print(df.dtypes)

Filas: 11000, Columnas: 17

--- Tipos de datos ---
age           int64
job          object
marital      object
education    object
default      object
balance       int64
housing      object
loan         object
contact      object
day           int64
month        object
duration      int64
campaign      int64
pdays         int64
previous      int64
poutcome     object
deposit      object
dtype: object


In [5]:
# ── 2.2 Clasificación de variables ───────────────────────────────────────
TARGET = 'deposit'
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

# Quitar target de las listas si estuviera
if TARGET in num_cols: num_cols.remove(TARGET)
if TARGET in cat_cols: cat_cols.remove(TARGET)

print(f'Variables NUMÉRICAS ({len(num_cols)}): {num_cols}')
print(f'Variables CATEGÓRICAS ({len(cat_cols)}): {cat_cols}')
print(f'Variable objetivo: {TARGET} (tipo: {df[TARGET].dtype})')

Variables NUMÉRICAS (7): ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
Variables CATEGÓRICAS (9): ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
Variable objetivo: deposit (tipo: object)


In [6]:
# ── 2.3 Alta cardinalidad (> 10 valores únicos) ───────────────────────────
print('\n--- Cardinalidad variables categóricas ---')
card_df = pd.DataFrame({
    'variable': cat_cols,
    'n_unique': [df[c].nunique() for c in cat_cols],
    'valores': [list(df[c].unique()) for c in cat_cols]
}).sort_values('n_unique', ascending=False)
print(card_df.to_string(index=False))

alta_card = card_df[card_df['n_unique'] > 10]['variable'].tolist()
print(f'\nVariables con alta cardinalidad (>10): {alta_card if alta_card else "Ninguna"}')


--- Cardinalidad variables categóricas ---
 variable  n_unique                                                                                                                                      valores
      job        12 [admin., technician, services, management, None, blue-collar, unemployed, entrepreneur, housemaid, retired, unknown, self-employed, student]
    month        12                                                                                 [may, jun, jul, aug, oct, nov, dec, jan, feb, mar, apr, sep]
education         4                                                                                                      [secondary, tertiary, primary, unknown]
 poutcome         4                                                                                                           [unknown, other, failure, success]
  marital         3                                                                                                            [married, single, divorc

In [7]:
# ── 2.4 Valores faltantes ─────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing': missing, 'pct': missing_pct})
missing_df = missing_df[missing_df['missing'] > 0].sort_values('missing', ascending=False)
if missing_df.empty:
    print('No hay valores faltantes en el dataset.')
else:
    print('Variables con valores faltantes:')
    print(missing_df)

Variables con valores faltantes:
         missing   pct
marital      282  2.56
job           97  0.88


In [8]:
# ── 2.5 Columnas constantes o de ID ──────────────────────────────────────
constant_cols = [c for c in df.columns if df[c].nunique() <= 1]
print(f'Columnas constantes: {constant_cols if constant_cols else "Ninguna"}')

# Heurística ID: nombres con 'id' y alta cardinalidad
id_cols = [c for c in df.columns if 'id' in c.lower() and df[c].nunique() == len(df)]
print(f'Posibles columnas ID: {id_cols if id_cols else "Ninguna"}')

Columnas constantes: Ninguna
Posibles columnas ID: Ninguna
